# ДЗ5: Класифікація зображень Intel Image Classification за допомогою CNN та Transfer Learning (PyTorch)

In [ ]:
import os
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset

from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import kagglehub

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Підготовка даних

Завантажуємо датасет **Intel Image Classification** з Kaggle. Він містить ~25 000 зображень розміром 150×150 пікселів, розподілених по 6 класах:
- `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`

Датасет вже розділений на `seg_train` та `seg_test`. Для валідації виділимо 20% тренувальних зображень.

In [ ]:
dataset_path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print(f"Dataset path: {dataset_path}")


def find_split_dir(base, name):
    """Find a directory named `name` or `name/name` under `base`."""
    candidate = os.path.join(base, name)
    if os.path.isdir(candidate):
        nested = os.path.join(candidate, name)
        return nested if os.path.isdir(nested) else candidate
    for root, dirs, _ in os.walk(base):
        if name in dirs:
            return os.path.join(root, name)
    raise FileNotFoundError(f"Directory '{name}' not found under {base}")


train_dir = find_split_dir(dataset_path, "seg_train")
test_dir = find_split_dir(dataset_path, "seg_test")
print(f"Train dir: {train_dir}")
print(f"Test dir : {test_dir}")

CLASS_NAMES = sorted(os.listdir(train_dir))
print(f"\nClasses ({len(CLASS_NAMES)}): {CLASS_NAMES}")

print("\nImage counts per class:")
print(f"{'Class':<12} {'Train':>8} {'Test':>8}")
print("-" * 30)
for cls in CLASS_NAMES:
    n_train = len(os.listdir(os.path.join(train_dir, cls)))
    n_test = len(os.listdir(os.path.join(test_dir, cls)))
    print(f"{cls:<12} {n_train:>8} {n_test:>8}")

In [ ]:
IMG_SIZE_CNN = 150
IMG_SIZE_RESNET = 224
BATCH_SIZE = 64

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_transforms(img_size):
    """Return train and val/test transform pipelines for a given image size."""
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_transform, val_transform

In [ ]:
def create_dataloaders(img_size, batch_size=BATCH_SIZE):
    """Create train, val, test DataLoaders for a given image size."""
    train_tf, val_tf = get_transforms(img_size)

    full_train_aug = ImageFolder(train_dir, transform=train_tf)
    full_train_clean = ImageFolder(train_dir, transform=val_tf)

    n_total = len(full_train_aug)
    n_val = int(0.2 * n_total)
    n_train = n_total - n_val

    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(n_total, generator=generator).tolist()
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]

    train_subset = Subset(full_train_aug, train_indices)
    val_subset = Subset(full_train_clean, val_indices)

    test_dataset = ImageFolder(test_dir, transform=val_tf)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    print(f"Image size: {img_size}x{img_size}")
    print(f"Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(test_dataset)}")

    return train_loader, val_loader, test_loader


train_loader_cnn, val_loader_cnn, test_loader_cnn = create_dataloaders(IMG_SIZE_CNN)

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for display."""
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor.clamp(0, 1)


images, labels = next(iter(train_loader_cnn))
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
shown = {c: 0 for c in range(len(CLASS_NAMES))}
idx = 0
for ax in axes.flat:
    while idx < len(labels) and shown[labels[idx].item()] >= 2:
        idx += 1
    if idx >= len(labels):
        ax.axis("off")
        continue
    img = denormalize(images[idx])
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[labels[idx].item()])
    ax.axis("off")
    shown[labels[idx].item()] += 1
    idx += 1
plt.suptitle("Зразки зображень з тренувального набору", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Варіант А: Проста згорткова нейронна мережа (CNN)

Створюємо власну CNN з 4 згортковими блоками (Conv2d → ReLU → MaxPool2d) та повнозв'язаним класифікатором. Кожен блок подвоює кількість фільтрів: 32 → 64 → 128 → 128. Після згорток — Flatten, FC(256), Dropout(0.5), FC(6).

In [ ]:
class SimpleCNN(nn.Module):
    """Custom CNN for Intel image classification (6 classes)."""

    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 9 * 9, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


cnn_model = SimpleCNN().to(device)
print(cnn_model)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 3. Функція втрат та оптимізатор

**CrossEntropyLoss** — стандартний вибір для задач багатокласової класифікації. Поєднує LogSoftmax і NLLLoss в одній функції, що дозволяє моделі повертати сирі logits без явного softmax. Ефективно оптимізує ймовірнісний розподіл по 6 класах, штрафуючи невпевненість у правильному класі.

**Adam** — адаптивний оптимізатор із моментами першого та другого порядків. Автоматично підбирає індивідуальний крок навчання для кожного параметра, що забезпечує стабільну збіжність без ретельного підбору learning rate.

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, print_every=5):
    """Train model with validation tracking and best-model checkpointing.

    Args:
        model: nn.Module to train
        train_loader: training DataLoader
        val_loader: validation DataLoader
        criterion: loss function
        optimizer: optimizer instance
        epochs: number of training epochs
        print_every: print progress every N epochs

    Returns:
        history: dict with train_loss, val_loss, train_acc, val_acc lists
    """
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss_sum += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

        if epoch % print_every == 0 or epoch == 1:
            print(f"Epoch {epoch:>3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    model.load_state_dict(best_weights)
    print(f"\nНайкраща валідаційна точність: {best_val_acc:.4f}")
    return history

In [ ]:
%%time
criterion = nn.CrossEntropyLoss()
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

EPOCHS_CNN = 25

print("=" * 65)
print("Навчання простої CNN: 4 conv blocks, Adam, lr=0.001, 25 epochs")
print("=" * 65)
cnn_history = train_model(
    cnn_model, train_loader_cnn, val_loader_cnn,
    criterion, cnn_optimizer, epochs=EPOCHS_CNN, print_every=5
)

## 4. Оцінка моделі

**Вибір метрик:**
- **Accuracy** — частка правильно класифікованих зображень. Підходить як базова метрика, оскільки класи у датасеті відносно збалансовані.
- **F1-score (weighted)** — гармонічне середнє precision та recall, зважене за кількістю зразків у кожному класі. Надійніша за accuracy при наявності навіть незначного дисбалансу класів, враховує як повноту, так і точність по кожному класу.

In [ ]:
def evaluate_model(model, test_loader, class_names, model_name="Model"):
    """Evaluate a trained classifier on the test set.

    Args:
        model: trained nn.Module
        test_loader: DataLoader for evaluation
        class_names: list of class name strings
        model_name: label for display

    Returns:
        dict with model, accuracy, f1, y_true, y_pred
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")

    print(f"\n{model_name}")
    print("-" * 50)
    print(f"  Accuracy     : {acc:.4f}")
    print(f"  F1 (weighted): {f1:.4f}")
    print(f"\n{classification_report(all_labels, all_preds, target_names=class_names)}")

    return {"model": model_name, "accuracy": acc, "f1": f1,
            "y_true": all_labels, "y_pred": all_preds}

In [ ]:
cnn_results = evaluate_model(cnn_model, test_loader_cnn, CLASS_NAMES, "Проста CNN")

## 5. Аналіз результатів простої CNN

In [ ]:
def plot_learning_curves(history, title):
    """Plot training and validation loss/accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Validation")
    axes[0].set_xlabel("Епоха")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{title}: Втрати")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Train")
    axes[1].plot(history["val_acc"], label="Validation")
    axes[1].set_xlabel("Епоха")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title(f"{title}: Точність")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names, title):
    """Plot confusion matrix as a seaborn heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Передбачений клас")
    ax.set_ylabel("Справжній клас")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_learning_curves(cnn_history, "Проста CNN")
plot_confusion_matrix(cnn_results["y_true"], cnn_results["y_pred"],
                      CLASS_NAMES, "Проста CNN: Матриця плутанини")

## 6. Варіант Б: Transfer Learning (ResNet18)

Використовуємо попередньо навчену на ImageNet модель ResNet18. Заморожуємо всі шари backbone і замінюємо останній повнозв'язаний шар на новий Linear(512, 6) для наших 6 класів. Це дозволяє використати потужні візуальні ознаки, навчені на мільйонах зображень, і швидко адаптувати модель до нового завдання з мінімальним ризиком перенавчання.

In [ ]:
train_loader_resnet, val_loader_resnet, test_loader_resnet = create_dataloaders(IMG_SIZE_RESNET)

In [ ]:
resnet_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in resnet_model.parameters():
    param.requires_grad = False

num_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_features, 6)

resnet_model = resnet_model.to(device)

trainable = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet_model.parameters())
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters   : {total - trainable:,}")

In [ ]:
%%time
resnet_optimizer = torch.optim.Adam(resnet_model.fc.parameters(), lr=0.001)

EPOCHS_RESNET = 10

print("=" * 65)
print("Навчання ResNet18 (transfer learning): frozen backbone, 10 epochs")
print("=" * 65)
resnet_history = train_model(
    resnet_model, train_loader_resnet, val_loader_resnet,
    criterion, resnet_optimizer, epochs=EPOCHS_RESNET, print_every=2
)

In [ ]:
resnet_results = evaluate_model(
    resnet_model, test_loader_resnet, CLASS_NAMES, "ResNet18 (Transfer Learning)"
)

In [ ]:
plot_learning_curves(resnet_history, "ResNet18 (Transfer Learning)")
plot_confusion_matrix(resnet_results["y_true"], resnet_results["y_pred"],
                      CLASS_NAMES, "ResNet18: Матриця плутанини")

## 7. Аналіз помилок класифікації

Візуалізуємо зразки, на яких модель ResNet18 помилилася, щоб зрозуміти типові складнощі класифікації.

In [ ]:
errors = np.where(resnet_results["y_true"] != resnet_results["y_pred"])[0]
print(f"Помилок: {len(errors)} / {len(resnet_results['y_true'])}")

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(errors, min(12, len(errors)), replace=False)

test_dataset = test_loader_resnet.dataset
for i, ax in enumerate(axes.flat):
    if i < len(sample_idx):
        idx = sample_idx[i]
        img, _ = test_dataset[idx]
        img = denormalize(img)
        true_label = CLASS_NAMES[resnet_results["y_true"][idx]]
        pred_label = CLASS_NAMES[resnet_results["y_pred"][idx]]
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.set_title(f"True: {true_label}\nPred: {pred_label}", fontsize=9, color="red")
    ax.axis("off")

plt.suptitle("ResNet18: Помилково класифіковані зображення", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Порівняння моделей

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Модель": cnn_results["model"],
        "Accuracy": round(cnn_results["accuracy"], 4),
        "F1 (weighted)": round(cnn_results["f1"], 4),
    },
    {
        "Модель": resnet_results["model"],
        "Accuracy": round(resnet_results["accuracy"], 4),
        "F1 (weighted)": round(resnet_results["f1"], 4),
    },
])

print("\nПорівняльна таблиця результатів:")
print("=" * 60)
print(comparison_df.to_string(index=False))
print("=" * 60)
print("\nОрієнтовні цільові показники:")
print("  Проста CNN:        Accuracy 75-85%, F1 0.73-0.83")
print("  Transfer Learning: Accuracy 90-97%, F1 0.89-0.96")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
model_names = ["Проста CNN", "ResNet18 (TL)"]
x = range(len(model_names))

for i, (metric, title) in enumerate([("accuracy", "Accuracy"), ("f1", "F1 (weighted)")]):
    values = [cnn_results[metric], resnet_results[metric]]
    bars = axes[i].bar(x, values, color=["steelblue", "darkorange"], width=0.5)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(model_names)
    axes[i].set_ylabel(title)
    axes[i].set_title(f"Порівняння: {title}")
    axes[i].set_ylim(0, 1)
    for j, v in enumerate(values):
        axes[i].text(j, v + 0.02, f"{v:.4f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 9. Висновки

### Результати

**Проста CNN** (4 згорткові блоки, ~2M параметрів) — демонструє помірну точність (~75-85%), що є очікуваним результатом для мережі, навченої з нуля на відносно невеликому датасеті. Основні складнощі — розрізнення візуально схожих класів (glacier/mountain, buildings/street).

**ResNet18 (Transfer Learning)** (заморожений backbone, ~3K навчених параметрів) — значно перевершує просту CNN (~90-97%), підтверджуючи ефективність переносу навчання.

### Чому Transfer Learning працює краще

1. **Попередні знання** — ResNet18 навчена на ImageNet (1.2M зображень, 1000 класів) і вже вміє розпізнавати краї, текстури, форми та складніші візуальні патерни.
2. **Ефективність даних** — замість навчання мільйонів параметрів з нуля, ми адаптуємо лише ~3000 параметрів останнього шару.
3. **Регуляризація** — заморожування backbone запобігає перенавчанню на обмеженому датасеті (~14K навчальних зображень).

### Критичний аналіз простої CNN

- Архітектура з 4 conv блоками має обмежену receptive field — може не захоплювати глобальні паттерни зображень.
- Відсутність Batch Normalization уповільнює збіжність.
- Dropout лише у FC-частині — згорткові шари можуть перенавчатися.

### Шляхи покращення

1. **Fine-tuning** — розморозити останні 1-2 residual блоки ResNet і довчити з малим lr (~1e-5).
2. **Batch Normalization** — додати BN між conv та ReLU у простій CNN для стабільнішого навчання.
3. **Learning Rate Scheduler** — CosineAnnealingLR або ReduceLROnPlateau для адаптивного кроку.
4. **Сильніша аугментація** — RandomAffine, RandomPerspective, CutOut для кращої генералізації.
5. **Інші архітектури** — EfficientNet-B0 або MobileNetV3 для кращого балансу точності та швидкості.